# Test chunking strategies

Use this notebook after testing extraction. It compares the two project-specific chunking strategies:

- **Fixed-size chunking**: word-based chunks with optional overlap.
- **Paragraph-aware chunking**: keeps neighbouring source sections together when possible.

Place a small TXT, PDF, or DOCX file in `data/uploads/`, select the **Python (.venv)** kernel, and run the cells from top to bottom.

In [1]:
from pathlib import Path
import sys

# Make sure Python can find the project's rag folder.
project_root = Path.cwd().resolve()
if not (project_root / 'rag').is_dir():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.chunking import fixed_size_chunks, paragraph_aware_chunks
from rag.extractors import extract_document

upload_folder = project_root / 'data' / 'uploads'

In [2]:
# Choose a supported document from data/uploads/.
supported_extensions = {'.txt', '.pdf', '.docx'}
available_files = sorted(
    file_path
    for file_path in upload_folder.iterdir()
    if file_path.is_file() and file_path.suffix.lower() in supported_extensions
)

if not available_files:
    raise FileNotFoundError('Add a TXT, PDF, or DOCX file to data/uploads/, then run this cell again.')

for index, file_path in enumerate(available_files):
    print(f'{index}: {file_path.name}')

# Change 0 to use a different file from the list above.
file_to_test = available_files[0]
sections = extract_document(file_to_test)
print(f'\nExtracted source sections: {len(sections)}')

0: Project_Proposal.docx

Extracted source sections: 39


In [3]:
# These settings are the experimental variables you will compare later.
chunk_size = 80  # Maximum number of words in one chunk.
overlap = 20    # Repeated words between fixed-size chunks.

fixed_chunks = fixed_size_chunks(
    sections,
    chunk_size=chunk_size,
    overlap=overlap,
)

paragraph_chunks = paragraph_aware_chunks(
    sections,
    chunk_size=chunk_size,
)

print(f'Fixed-size chunks: {len(fixed_chunks)}')
print(f'Paragraph-aware chunks: {len(paragraph_chunks)}')

Fixed-size chunks: 39
Paragraph-aware chunks: 19


In [4]:
# Preview fixed-size chunks and their source metadata.
for chunk in fixed_chunks[:2]:
    print('=' * 80)
    print(f"{chunk['chunk_id']} | {chunk['word_count']} words")
    print(f"Source: {chunk['source_label']}")
    print(chunk['text'][:500])

Project_Proposal.docx:paragraph:1:chunk:1 | 2 words
Source: Project_Proposal.docx, paragraph 1
Project Proposal
Project_Proposal.docx:paragraph:2:chunk:1 | 12 words
Source: Project_Proposal.docx, paragraph 2
Ask My Documents: A RAG System for Question Answering over Uploaded Documents


In [5]:
# Preview paragraph-aware chunks. Notice when several source sections are combined.
for chunk in paragraph_chunks[:2]:
    print('=' * 80)
    print(f"{chunk['chunk_id']} | {chunk['word_count']} words")
    print(f"Sources combined: {chunk['source_count']}")
    print(f"Source: {chunk['source_label']}")
    print(chunk['text'][:500])

Project_Proposal.docx:paragraph:1:chunk:1 | 33 words
Sources combined: 4
Source: Project_Proposal.docx, paragraph 1 | Project_Proposal.docx, paragraph 2 | Project_Proposal.docx, paragraph 3 | Project_Proposal.docx, paragraph 5
Project Proposal

Ask My Documents: A RAG System for Question Answering over Uploaded Documents

Student: Emem Usoh | Supervisor: Professor Dongyu Qiu | Duration: Summer 2 (6 weeks)

1. Project Idea and Motivation
Project_Proposal.docx:paragraph:6:chunk:1 | 55 words
Sources combined: 1
Source: Project_Proposal.docx, paragraph 6
I propose to build Ask My Documents, a document question-answering application. A user will upload PDF, DOCX, and TXT files, ask questions about them, and receive answers based only on the uploaded documents. The system will also show the source document or passage used, so the answer can be checked instead of being accepted blindly.


In [6]:
# Basic checks: every chunk should fit the chosen word limit and keep citation metadata.
all_chunks = fixed_chunks + paragraph_chunks
assert all(chunk['word_count'] <= chunk_size for chunk in all_chunks)
assert all(chunk['source_ids'] for chunk in all_chunks)
assert all(chunk['source_label'] for chunk in all_chunks)

print('Chunking checks passed.')

Chunking checks passed.


## What to observe

- Fixed-size chunks may split one source section into several chunks and repeat words because of overlap.
- Paragraph-aware chunks may combine several short sections, but do not exceed the word limit.
- Every chunk retains source labels and IDs, which will later support citations.

Try the same document with different `chunk_size` and `overlap` values. Record the settings you use so they can become part of the evaluation experiments.